# RoboCasa Demo Gallery

一键启动 [RoboCasa](https://robocasa.ai) 厨房场景与任务的 MuJoCo viewer，外加官方预训练 GR00T N1.5 的 eval。

_One-click MuJoCo viewer for RoboCasa kitchen scenes & tasks, plus eval with the official pretrained GR00T N1.5._

**为什么单开一页 / Why a separate notebook**：RoboCasa 的 `setup.py` 把 `mujoco==3.3.1` / `numpy==2.2.5` 硬钉，跟主 `mujoco` env 里其他场景库（dm_control / mjlab / mujoco_playground 都要 mujoco≥3.7/3.8）注定冲突。所以放在独立的 conda env `robocasa` 里关起来养。GR00T 的依赖（`numpy==1.26.4`）又和 RoboCasa 冲突，所以 §5 eval 再多开一个 `robocasa_gr00t` env。
_RoboCasa hard-pins `mujoco==3.3.1` / `numpy==2.2.5`, colliding with the main `mujoco` env. GR00T pins `numpy==1.26.4`, colliding with RoboCasa. So we end up with two extra conda envs: `robocasa` (sim) + `robocasa_gr00t` (policy)._

**机制 / Mechanism**：所有逻辑在 `scripts/robocasa_demo.py` + `scripts/install_robocasa_env.sh` + `scripts/robocasa_eval_gr00t.py` + `scripts/install_gr00t_env.sh`。notebook 只通过 `!python ...` / `!bash ...` 调用。

**前置 / Prereqs**:
- `conda` 可用 / `conda` available
- 显示器 (`DISPLAY=:0`)（仅 viewer demo §1-§2 用 / only needed for viewer §1-§2）
- ~35-40 GB 磁盘（厨房资产 ~10 GB + robocasa env ~5 GB + GR00T env ~15 GB + N1.5 ckpt ~7 GB）
- NVIDIA GPU ≥8 GB（仅 §5 eval 用 / only for §5 eval）


## 0. 安装 conda env `robocasa` (Setup)

**幂等 / Idempotent**：env / robocasa / 资产 都已存在则跳过。

**做了什么 / What it does**（见 `scripts/install_robocasa_env.sh`）:
1. `conda create -n robocasa python=3.11`
2. `pip install git+https://github.com/ARISE-Initiative/robosuite.git@master`（不是 PyPI 版！RoboCasa 需要最新 master）
3. `pip install -e dependencies/robocasa`
4. 生成 `macros_private.py`，让 `DATASET_BASE_PATH` 指向 `$ROBOCASA_DATA_PATH/datasets`
5. `python -m robocasa.scripts.download_kitchen_assets --type all` （~10 GB，纹理 + 家具 + objaverse 物体）
6. 写 activate hook 持久化 `ROBOCASA_DATA_PATH=~/.cache/robocasa`

首次跑约 15-30 分钟（取决于带宽）。


In [ ]:
# 一键安装 conda env `robocasa` + 所有依赖 + ~23GB 厨房资产。幂等。
# This single command runs scripts/install_robocasa_env.sh which does:
#
#   conda create -c conda-forge -n robocasa python=3.11 -y
#   conda activate robocasa
#   pip install git+https://github.com/ARISE-Initiative/robosuite.git@master
#   pip install -e dependencies/robocasa
#   pip install robosuite_models
#   python -m robocasa.scripts.download_kitchen_assets --type all   # ~23 GB
#   # + write macros_private.py and activate hook (ROBOCASA_DATA_PATH)
#
# 首次跑 15-30 分钟（取决于带宽）。已装好则跳过。
!bash scripts/install_robocasa_env.sh


In [ ]:
!python scripts/robocasa_demo.py status


In [ ]:
!python scripts/robocasa_demo.py list


---
## 0.5 GR00T 策略准备 (Policy Setup) — §2 跑前必装

§2 的操作任务用 **GR00T N1.5 atomic-seen post-trained** checkpoint 实跑。需要：

1. **`robocasa_gr00t` conda env**（torch 2.5.1 + flash-attn，跟 robocasa env 不能共用因为 numpy 版本冲突）— ~15-20 GB，**首次 15-30 min**
2. **atomic-seen post-trained checkpoint**（推理子集 7.1 GB，paper avg success rate 68.5%）— ~2 min 下完

幂等：装过 / 下过会跳过。


In [ ]:
# 1) 装 robocasa_gr00t env (~15-20 GB, 15-30 min)
# 内部做的事见 scripts/install_gr00t_env.sh
!bash scripts/install_gr00t_env.sh


In [ ]:
# 2) 下 atomic-seen post-trained checkpoint (~7.1 GB, ~2 min)
# 砍掉 optimizer/scheduler，只下推理用的 safetensors + config + experiment_cfg
!source $(conda info --base)/etc/profile.d/conda.sh && conda activate robocasa && python - <<'PY'
import os
from huggingface_hub import snapshot_download
base = os.environ.get("ROBOCASA_DATA_PATH", os.path.expanduser("~/.cache/robocasa"))
target = os.path.join(base, "checkpoints")
ckpt = "gr00t_n1-5/foundation_model_learning/target_posttraining/atomic_seen/checkpoint-60000"
print(f">>> downloading {ckpt} to {target}")
snapshot_download(
    repo_id="robocasa/robocasa365_checkpoints",
    repo_type="model",
    local_dir=target,
    allow_patterns=[
        f"{ckpt}/model-*.safetensors",
        f"{ckpt}/model.safetensors.index.json",
        f"{ckpt}/config.json",
        f"{ckpt}/experiment_cfg/*",
    ],
    max_workers=4,
)
print(">>> done")
PY
!du -sh ~/.cache/robocasa/checkpoints/gr00t_n1-5/foundation_model_learning/target_posttraining/atomic_seen/checkpoint-60000 2>/dev/null


---
## 1. 厨房场景浏览 (Kitchen Scene Browsing)

加载不同 `(layout, style)` 组合的厨房场景。viewer 启动后机器人停在原地（idle），鼠标拖动视角自由观察。按 `Ctrl-C` 退出。

_Different kitchen layouts × styles. Robot idles; drag mouse to orbit camera. Ctrl-C to quit._


In [ ]:
# 标准厨房 (layout 1, style 1) + PandaOmron 移动操作平台
!python scripts/robocasa_demo.py launch scene:browse


In [ ]:
# 中岛厨房 (layout 3, style 2)
!python scripts/robocasa_demo.py launch scene:island


In [ ]:
# 一字型厨房 (layout 4, style 4)
!python scripts/robocasa_demo.py launch scene:galley


In [ ]:
# 随机布局 + 随机风格（每次 reset 不一样）
!python scripts/robocasa_demo.py launch scene:random


---
## 2. 操作任务 — 用 GR00T 策略实跑 (Manipulation Tasks with GR00T Policy)

下面是 **atomic-seen 全 18 个任务**（atomic-seen post-trained checkpoint 训练时覆盖范围），让 GR00T N1.5 真去操作。Cell 按**难度分三组**，建议从易到难跑。

_All 18 atomic-seen tasks (the full coverage of the atomic-seen post-trained checkpoint). Grouped by difficulty — start from §2.1 easy._

**前提**：上面 **§0.5** 已完成（装 `robocasa_gr00t` env + 下 atomic-seen post-trained ckpt）。每 cell 默认 5 episodes + `--render`。

**任务 / 难度 / 5-ep 期望成功数**：

| §  | 任务 | 难度 | 期望命中 |
|---|---|---|---|
| 2.1 易 | TurnOnSinkFaucet | 转水龙头 | 3-5 |
|  | CloseFridge | 关冰箱门 | 3-5 |
|  | CloseBlenderLid | 关搅拌机盖 | 3-5 |
|  | CloseToasterOvenDoor | 关烤箱门 | 3-5 |
|  | TurnOnElectricKettle | 开电水壶 | 3-5 |
|  | TurnOnMicrowave | 开微波炉 | 2-4 |
|  | TurnOffStove | 关炉灶 | 2-4 |
| 2.2 中 | OpenCabinet | 开柜门 | 2-3 |
|  | OpenDrawer | 开抽屉 | 2-3 |
|  | OpenStandMixerHead | 抬立式搅拌机头 | 2-3 |
|  | SlideDishwasherRack | 拉洗碗机架 | 2-3 |
|  | CoffeeSetupMug | 放杯到咖啡机 | 1-3 |
|  | NavigateKitchen | 导航到目标点 | 2-3 |
| 2.3 难 | PickPlaceCounterToCabinet | 台面→柜 | 1-2 |
|  | PickPlaceCounterToStove | 台面→灶 | 1-2 |
|  | PickPlaceDrawerToCounter | 抽屉→台面 | 1-2 |
|  | PickPlaceSinkToCounter | 水槽→台面 | 1-2 |
|  | PickPlaceToasterToCounter | 烤箱→台面 | 1-2 |

paper 报告 atomic-seen 平均 SR **68.5%**；我们实测 TurnOnSinkFaucet 5/5 通、10-ep 70%。


In [ ]:
# 拾放：台面 → 柜子（atomic_seen 里偏难，预期 1-2/5）
!python scripts/robocasa_eval_gr00t.py \
  --env-name PickPlaceCounterToCabinet \
  --ckpt checkpoints/gr00t_n1-5/foundation_model_learning/target_posttraining/atomic_seen/checkpoint-60000 \
  --n-episodes 5 --max-steps 400 --render --render-warmup-s 6


In [ ]:
# 拾放：水槽 → 台面（双臂 pick-and-place，预期 1-2/5）
!python scripts/robocasa_eval_gr00t.py \
  --env-name PickPlaceSinkToCounter \
  --ckpt checkpoints/gr00t_n1-5/foundation_model_learning/target_posttraining/atomic_seen/checkpoint-60000 \
  --n-episodes 5 --max-steps 400 --render --render-warmup-s 6


In [ ]:
# 开柜门（预期 2-3/5）
!python scripts/robocasa_eval_gr00t.py \
  --env-name OpenCabinet \
  --ckpt checkpoints/gr00t_n1-5/foundation_model_learning/target_posttraining/atomic_seen/checkpoint-60000 \
  --n-episodes 5 --max-steps 400 --render --render-warmup-s 6


In [ ]:
# 关冰箱门（atomic_seen 里最容易之一，预期 3-5/5）
!python scripts/robocasa_eval_gr00t.py \
  --env-name CloseFridge \
  --ckpt checkpoints/gr00t_n1-5/foundation_model_learning/target_posttraining/atomic_seen/checkpoint-60000 \
  --n-episodes 5 --max-steps 400 --render --render-warmup-s 6


In [ ]:
# 关炉灶（预期 2-4/5）
!python scripts/robocasa_eval_gr00t.py \
  --env-name TurnOffStove \
  --ckpt checkpoints/gr00t_n1-5/foundation_model_learning/target_posttraining/atomic_seen/checkpoint-60000 \
  --n-episodes 5 --max-steps 400 --render --render-warmup-s 6


In [ ]:
# 开水龙头（atomic_seen 里命中率最高之一，预期 3-5/5）
!python scripts/robocasa_eval_gr00t.py \
  --env-name TurnOnSinkFaucet \
  --ckpt checkpoints/gr00t_n1-5/foundation_model_learning/target_posttraining/atomic_seen/checkpoint-60000 \
  --n-episodes 5 --max-steps 400 --render --render-warmup-s 6


In [ ]:
# 开微波炉（预期 2-4/5）
!python scripts/robocasa_eval_gr00t.py \
  --env-name TurnOnMicrowave \
  --ckpt checkpoints/gr00t_n1-5/foundation_model_learning/target_posttraining/atomic_seen/checkpoint-60000 \
  --n-episodes 5 --max-steps 400 --render --render-warmup-s 6


In [ ]:
# 关搅拌机盖（atomic_seen 易，预期 3-5/5）
!python scripts/robocasa_eval_gr00t.py \
  --env-name CloseBlenderLid \
  --ckpt checkpoints/gr00t_n1-5/foundation_model_learning/target_posttraining/atomic_seen/checkpoint-60000 \
  --n-episodes 5 --max-steps 400 --render --render-warmup-s 6


In [ ]:
# 关烤箱门（易，预期 3-5/5）
!python scripts/robocasa_eval_gr00t.py \
  --env-name CloseToasterOvenDoor \
  --ckpt checkpoints/gr00t_n1-5/foundation_model_learning/target_posttraining/atomic_seen/checkpoint-60000 \
  --n-episodes 5 --max-steps 400 --render --render-warmup-s 6


In [ ]:
# 开电水壶（易，预期 3-5/5）
!python scripts/robocasa_eval_gr00t.py \
  --env-name TurnOnElectricKettle \
  --ckpt checkpoints/gr00t_n1-5/foundation_model_learning/target_posttraining/atomic_seen/checkpoint-60000 \
  --n-episodes 5 --max-steps 400 --render --render-warmup-s 6


In [ ]:
# 开抽屉（中，预期 2-3/5）
!python scripts/robocasa_eval_gr00t.py \
  --env-name OpenDrawer \
  --ckpt checkpoints/gr00t_n1-5/foundation_model_learning/target_posttraining/atomic_seen/checkpoint-60000 \
  --n-episodes 5 --max-steps 400 --render --render-warmup-s 6


In [ ]:
# 抬立式搅拌机头（中，预期 2-3/5）
!python scripts/robocasa_eval_gr00t.py \
  --env-name OpenStandMixerHead \
  --ckpt checkpoints/gr00t_n1-5/foundation_model_learning/target_posttraining/atomic_seen/checkpoint-60000 \
  --n-episodes 5 --max-steps 400 --render --render-warmup-s 6


In [ ]:
# 拉洗碗机架（中，预期 2-3/5）
!python scripts/robocasa_eval_gr00t.py \
  --env-name SlideDishwasherRack \
  --ckpt checkpoints/gr00t_n1-5/foundation_model_learning/target_posttraining/atomic_seen/checkpoint-60000 \
  --n-episodes 5 --max-steps 400 --render --render-warmup-s 6


In [ ]:
# 放杯到咖啡机（中，预期 1-3/5）
!python scripts/robocasa_eval_gr00t.py \
  --env-name CoffeeSetupMug \
  --ckpt checkpoints/gr00t_n1-5/foundation_model_learning/target_posttraining/atomic_seen/checkpoint-60000 \
  --n-episodes 5 --max-steps 400 --render --render-warmup-s 6


In [ ]:
# 导航到目标点（中，预期 2-3/5）— 看底盘移动
!python scripts/robocasa_eval_gr00t.py \
  --env-name NavigateKitchen \
  --ckpt checkpoints/gr00t_n1-5/foundation_model_learning/target_posttraining/atomic_seen/checkpoint-60000 \
  --n-episodes 5 --max-steps 400 --render --render-warmup-s 6


In [ ]:
# 拾放：台面 → 灶台（难，预期 1-2/5）
!python scripts/robocasa_eval_gr00t.py \
  --env-name PickPlaceCounterToStove \
  --ckpt checkpoints/gr00t_n1-5/foundation_model_learning/target_posttraining/atomic_seen/checkpoint-60000 \
  --n-episodes 5 --max-steps 400 --render --render-warmup-s 6


In [ ]:
# 拾放：抽屉 → 台面（难，预期 1-2/5）
!python scripts/robocasa_eval_gr00t.py \
  --env-name PickPlaceDrawerToCounter \
  --ckpt checkpoints/gr00t_n1-5/foundation_model_learning/target_posttraining/atomic_seen/checkpoint-60000 \
  --n-episodes 5 --max-steps 400 --render --render-warmup-s 6


In [ ]:
# 拾放：烤箱 → 台面（难，预期 1-2/5）
!python scripts/robocasa_eval_gr00t.py \
  --env-name PickPlaceToasterToCounter \
  --ckpt checkpoints/gr00t_n1-5/foundation_model_learning/target_posttraining/atomic_seen/checkpoint-60000 \
  --n-episodes 5 --max-steps 400 --render --render-warmup-s 6


---
## 2.5 π0.5 对照评估 (Pi0.5 Comparison Eval)

跑 [Physical Intelligence π0.5](https://www.physicalintelligence.company/blog/pi05) 在 RoboCasa 上的官方 ckpt（`pi05_pretrain_human300`），跟 §2 的 GR00T 形成对照。Leaderboard 报 atomic-seen **39.6%**（vs GR00T N1.5 50.7% / 我们 §2 post-trained 70%）。

_Run the official Physical Intelligence π0.5 ckpt on RoboCasa for direct comparison. Leaderboard: atomic-seen 39.6% (vs GR00T N1.5 50.7%; our post-trained 70% in §2)._

**为什么单开**：π0.5 的官方 ckpt 是 **Orbax/JAX 格式**（10 GB），上游 openpi 没有 RoboCasa 支持，我们在 [`scripts/_pi05_inference_server.py`](./scripts/_pi05_inference_server.py) 自定义 data config 把它接到 robocasa env。两进程架构跟 §2 GR00T 一样（一个 conda env 跑 sim，一个 uv venv 跑 JAX 推理）。

**前置**：§0 已经装好 `robocasa` env。下面 §2.5.1-§2.5.4 全部幂等。

**踩坑提醒（已在 server 里处理好）**：
- `discrete_state_input=True`（pi05 native default，**不能**照搬 pi05_libero 的 False，否则 SR=0）
- `XLA_FLAGS=--xla_gpu_autotune_level=0`（RTX 4090 + JAX 0.5.3 cuda12 wheel 会触发 ptxas SIGSEGV）
- norm_stats 只有 mean/std 没有 q01/q99 → 强制 `use_quantile_norm=False`
- `data_config.repack_transforms` 只训练用，推理要单独传 `repack_transforms=` 给 `create_trained_policy`

详见 `scripts/_pi05_inference_server.py` 顶部注释。

### §2.5.1 装 env + 下 ckpt


In [ ]:
# 1) 装 openpi 推理 env (clones dependencies/openpi + uv sync, 5-10 min)
# 这个 env 跟 GR00T env 不能共用 — openpi 钉 JAX 0.5.3 / Flax 0.10.2 / numpy<2.0，
# 跟 GR00T 的 torch 2.5.1 / numpy 1.26.4 互斥。所以三个并存的 env：
#   robocasa (sim, mujoco 3.3.1 + numpy 2.2.5)
#   robocasa_gr00t (GR00T inference, torch 2.5.1)
#   dependencies/openpi/.venv (π0.5 inference, JAX 0.5.3)
!bash scripts/install_pi05_env.sh


In [ ]:
# 2) 下载 π0.5 ckpt (~12 GB, 5-10 min)
# 只下 params/ + assets/，跳过 train_state/（~22 GB 优化器状态推理用不上）
!source $(conda info --base)/etc/profile.d/conda.sh && conda activate robocasa && python - <<'PY'
import os
from huggingface_hub import snapshot_download
base = os.environ.get("ROBOCASA_DATA_PATH", os.path.expanduser("~/.cache/robocasa"))
target = os.path.join(base, "checkpoints")
ckpt = "pi05_pretrain_human300/multitask_learning/75000"
print(f">>> downloading {ckpt}")
snapshot_download(
    repo_id="robocasa/robocasa365_checkpoints",
    repo_type="model",
    local_dir=target,
    allow_patterns=[
        f"{ckpt}/params/**",
        f"{ckpt}/assets/**",
        f"{ckpt}/_CHECKPOINT_METADATA",
    ],
    max_workers=8,
)
print(">>> done")
PY
!du -sh ~/.cache/robocasa/checkpoints/pi05_pretrain_human300/multitask_learning/75000 2>/dev/null


### §2.5.2 渲染预览 + 统计 SR

第一段：单 episode + viewer 看 π0.5 实际行为（对照 §2 同任务的 GR00T）。第二段：10 episode 无 render 出统计 SR（验证 leaderboard 39.6%）。

_Render preview (compare to GR00T §2) + 10-ep headless SR baseline._


In [ ]:
# 渲染预览：开柜门，单 episode + viewer。π0.5 比 GR00T 弱不少，可能不一次就完成 —
# 这里目的是看实际行为（接近完成 vs 漫无目的）。Ctrl-C 提前退出。
!python scripts/robocasa_eval_pi05.py \
  --env-name OpenCabinet \
  --split pretrain \
  --n-episodes 1 --max-steps 400 \
  --discrete-state-input \
  --render --render-warmup-s 6


In [ ]:
# 10 episodes 无 render baseline，对照 leaderboard 报的 pi0.5 atomic-seen 39.6%
# (我们实测 OpenCabinet pretrain 10/10 = 40%, 跟 leaderboard 几乎一致)
!python scripts/robocasa_eval_pi05.py \
  --env-name OpenCabinet \
  --split pretrain \
  --n-episodes 10 --max-steps 400 \
  --discrete-state-input


---
## 3. 关于其它机器人 (Other Robots)

理论上 RoboCasa 还支持 **GR1（人形）/ Tiago（移动操作）**，但当前 robosuite master 的 GR1 控制器配置引用了 `WHOLE_BODY_MINK_IK` 控制器类 —— 该类在仓库里**还没实现**，安装 `mink` / `robosuite_models` 都救不了，是上游 bug。Tiago 加载到 `mj_forward` 时报 `FactorizeHessian: rank-deficient sparse Hessian`，也是模型问题。

👉 现阶段 RoboCasa 在本仓的可用机器人是 **PandaOmron**（Franka Panda + Omron LD-60 移动底盘），已经覆盖上面所有场景与任务 demo。等 robosuite 上游把 mink 控制器补全后再开 GR1/Tiago。

_GR1 and Tiago hit upstream robosuite bugs (missing `WHOLE_BODY_MINK_IK` controller class, broken Tiago model). PandaOmron is the only working robot for now._


---
## 5. 进阶：跑大规模 baseline (Advanced: Full Statistical Eval)

§2 是单任务 demo（5 ep + render）。这里跑无 render 的大批 episode 验证 paper 数字（atomic-seen 平均 SR 68.5%）。


### 5.1 任务列表 + 多 episode SR baseline

`pretrain` split = 模型见过的 2500 个厨房；`target` split = 没见过的 10 个 holdout 厨房（难度大幅上升）。

_`pretrain` split = 2500 seen kitchens, `target` split = 10 held-out kitchens (much harder)._


In [ ]:
# 列出所有 task set，看每个 set 包含哪些任务
!source $(conda info --base)/etc/profile.d/conda.sh && conda activate robocasa && python - <<'PY'
from robocasa.utils.dataset_registry import TASK_SET_REGISTRY
for k in ("atomic_seen", "composite_seen", "composite_unseen"):
    print(f"\n=== {k}  ({len(TASK_SET_REGISTRY[k])} tasks) ===")
    for t in TASK_SET_REGISTRY[k]:
        print(f"  {t}")
PY


In [ ]:
# 10 episodes 无 render 跑 baseline，验证 paper 报的 68.5% (我们实测 ~70%)
!python scripts/robocasa_eval_gr00t.py \
  --env-name TurnOnSinkFaucet \
  --ckpt checkpoints/gr00t_n1-5/foundation_model_learning/target_posttraining/atomic_seen/checkpoint-60000 \
  --n-episodes 10 --max-steps 400


---
### 5.2 官方 leaderboard 对照：DP / π0 / π0.5 / GR00T

数字来源：[robocasa.ai/leaderboard](https://robocasa.ai/leaderboard)（统一 eval setup，每方法用 `robocasa365_checkpoints` 仓里同一档 ckpt path），评测 50 target tasks。

_Numbers from the official leaderboard — same eval protocol, one ckpt path per method, 50 target tasks._

| Policy | Atomic-Seen | Composite-Seen | Composite-Unseen | Overall |
|---|---:|---:|---:|---:|
| **RLDX-1** (闭源 SOTA) | **63.0%** | **27.5%** | **5.4%** | **33.2%** |
| **GR00T N1.5** (本仓默认 family) | 50.7% | 14.8% | 2.7% | 23.9% |
| GR00T N1.6 | 51.1% | 9.4% | 1.7% | 21.9% |
| **π0.5** | 39.6% | 7.1% | 1.2% | 16.9% |
| π0 | 34.6% | 6.1% | 1.1% | 14.8% |
| **Diffusion Policy** | 15.7% | 0.2% | 1.3% | 6.1% |

**关键观察 / Key takeaways**：

1. **GR00T N1.5 ≈ 1.3× π0.5 ≈ 3.2× DP**（atomic-seen 50.7 / 39.6 / 15.7%）—— 同份 RoboCasa365 数据训出来，**模型规模决定基线高度**。
2. **composite 任务全员崩盘**（最高 RLDX-1 才 27.5%）—— 跨原子动作的长程组合是当前所有方法的共同硬伤。
3. **N1.6 总分反而 < N1.5**（21.9 vs 23.9）—— 更新版基座不一定更强，本仓默认 N1.5 没冤枉它。
4. **DP 仅 6.1%** —— 不带 VLM 的纯视觉策略在 50 任务多模态泛化上接近报废，证明 language conditioning + 大基座的必要性。

**与本仓 §2 / §5.1 实测数字的关系 / How this maps to our §2 / §5.1 numbers**：

- Leaderboard 上 GR00T N1.5 = **50.7%** atomic-seen，对应 ckpt 路径 `foundation_model_learning/target_only/atomic_seen/`（**target-only finetune**，没用 pretrain 阶段）
- 本仓默认 ckpt 是 `target_posttraining/atomic_seen/checkpoint-60000` —— **pretrain + posttrain 双阶段**，atomic-seen ≈ **68.5%**（paper Tab.3 / 本仓 §5.1 实测 ~70%）
- 这 +18 pts 的根因（pretrain 的物理常识 + 视觉表征迁移）详见 [`doc/robocasa_gr00t_checkpoints.html`](./doc/robocasa_gr00t_checkpoints.html)

_The 50.7% leaderboard number is the target-only baseline. Our default `target_posttraining/atomic_seen/checkpoint-60000` is the post-trained variant (paper 68.5%, our §5.1 ~70%) — same model family, different fine-tune stage._

**结论 / Bottom line**：在 RoboCasa365 公开 ckpt 里，**GR00T N1.5 post-trained 是公开可用的最强 baseline**。π0.5 差 ~11 pts，DP 差 ~35 pts，没有可直接替代的小模型对照。


---
## 4. 收尾 (Cleanup)


In [ ]:
!python scripts/robocasa_demo.py kill


---
## 进一步 / Next Steps

- **训练数据**：`python -m robocasa.scripts.download_datasets --tasks PnPCounterToCab` 下载该任务的 human/robot demos（每个任务几 GB）
- **回放轨迹**：`python -m robocasa.demos.demo_tasks` 选任务后自动下载并回放 demonstrations（交互式）
- **遥操作收集**：`python -m robocasa.demos.demo_teleop --task Kitchen` 键盘控制 PandaOmron，录制自己的 demo
- **更多 checkpoint**：除了 §5 的 GR00T N1.5 multitask，[robocasa365_checkpoints](https://huggingface.co/robocasa/robocasa365_checkpoints) 还有 Diffusion Policy / π0 / π0.5 / GR00T 的 `foundation_model_learning` 和 `lifelong_learning` 分支（总计 282 GB；按需挑 `checkpoint-*/` 子目录单独 `snapshot_download`）
- **场景列表**：60 种 layouts × 12 种 styles = 720 种厨房组合
